# CESM2-LE ENSO CP/TP Indices — Exploration

Plots **Cold-Tongue (N_CT)** and **Warm-Pool (N_WP)** ENSO indices for a selected CESM2-LE ensemble member, following Takahashi et al.:

```
N_CT = N3 − α × N4
N_WP = N4 − α × N3
α = 2/5  if N3 × N4 > 0,  else  0
```

All indices (Niño3, Niño4, Niño3.4, N_CT, N_WP) are 5-month smoothed and normalised.

**Plots produced:**
1. Full time series for all CP/TP indices — all months, all years
2. Time series for a selected calendar month only
3. SST composite maps for N_CT+, N_CT−, N_WP+, N_WP− phases

In [ ]:
import sys
from pathlib import Path
import numpy as np
import xarray as xr
import netCDF4 as nc
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
try:
    import cmocean
    CMAP_SST = cmocean.cm.balance
except ImportError:
    CMAP_SST = 'RdBu_r'

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))
from configs import paths

MONTH_LABELS = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
MONTH_NAMES  = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

## 1. User Parameters
Edit the cell below before running the notebook.

In [ ]:
# ─────────────────────────────────────────────────
# USER PARAMETERS
# ─────────────────────────────────────────────────

ens_member  = 0     # Ensemble member index (0–99)
                    #   0–49  → CMIP6 members
                    #  50–99  → SMBB members

plot_month  = 9     # Calendar month for the monthly time-series plot
                    # (1 = Jan, 9 = Sep, 12 = Dec)

# Composite map settings
clim_vmin, clim_vmax = -0.5, 0.5    # SST anomaly colorbar limits (K)

# File locations
CPTP_FILE = paths.CESM2LE_DIR / 'climate_indices' / 'cesm2le_enso_cptp_indices.nc'
SST_DIR   = paths.CESM2LE_SST_DIR / 'mon'
GRID_FILE = paths.CESM2LE_SST_DIR / 'grid' / 'cesm2le_sst_grid.nc'

In [ ]:
import matplotlib.patches as mpatches

# ── ENSO CP/TP region boxes ─────────────────────────────────────────────────
#   All 5°S–5°N; longitude in 0–360°E to match climate_indices.py
CPTP_BOXES = {
    'Niño3.4  (170°W–120°W)': dict(latmin=-5, latmax=5, lonmin=190, lonmax=240, color='tab:red'),
    'Niño3    (150°W–90°W)':  dict(latmin=-5, latmax=5, lonmin=210, lonmax=270, color='tab:orange'),
    'Niño4    (160°E–150°W)': dict(latmin=-5, latmax=5, lonmin=160, lonmax=210, color='tab:blue'),
}

proj  = ccrs.PlateCarree(central_longitude=180)
trans = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=(14, 4), subplot_kw={'projection': proj})
ax.set_global()
ax.coastlines(linewidth=0.6, color='0.4')
ax.set_facecolor('white')

handles = []
for label, b in CPTP_BOXES.items():
    rect = mpatches.Rectangle(
        (b['lonmin'], b['latmin']),
        b['lonmax'] - b['lonmin'],
        b['latmax'] - b['latmin'],
        linewidth=2, edgecolor=b['color'], facecolor=b['color'],
        alpha=0.3, transform=trans, zorder=3
    )
    ax.add_patch(rect)
    handles.append(mpatches.Patch(facecolor=b['color'], alpha=0.5, label=label))

gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

ax.legend(handles=handles, loc='lower left', fontsize=9, framealpha=0.9)
ax.set_title('ENSO CP/TP index boxes  (N_CT = N3 − α·N4,  N_WP = N4 − α·N3)', fontsize=12, pad=8)
fig.tight_layout()
plt.show()

## 2. Load ENSO CP/TP Data

In [ ]:
ds_cptp = xr.open_dataset(CPTP_FILE)
print(ds_cptp)

years = ds_cptp['nyr'].values    # (nyear,)
nyear = len(years)

# Available index names in the file (shaped arrays only, no _flat suffix)
index_keys  = ['n34', 'n3', 'n4', 'n_ct', 'n_wp']
label_keys  = ['labels_n34', 'labels_n3', 'labels_n4', 'labels_n_ct', 'labels_n_wp']

# Pull the arrays for the selected member — shape (nyear, 12)
indices = {k: ds_cptp[k].values[ens_member] for k in index_keys}
labels  = {k: ds_cptp[k].values[ens_member] for k in label_keys}

# Flat time series
indices_flat = {k: v.reshape(-1) for k, v in indices.items()}
labels_flat  = {k: v.reshape(-1) for k, v in labels.items()}

# Datetime index
dates = pd.date_range(start=f'{years[0]}-01', periods=nyear * 12, freq='MS')

print(f'\nEnsemble member : {ens_member}')
print(f'Years           : {years[0]}–{years[-1]}  ({nyear} years, {len(dates)} months)')
for k in label_keys:
    lf = labels_flat[k]
    print(f'  {k:16s}  +1: {(lf==1).sum():4d},  0: {(lf==0).sum():4d},  -1: {(lf==-1).sum():4d}')

## 3. Time Series — All Months

Full monthly time series for all five indices.  
Red shading = positive phase, blue shading = negative phase (using Niño3.4 labels for the top panel; N_CT / N_WP labels for the lower panels).

In [ ]:
def shade_phases(ax, dates, phases, alpha=0.15):
    start = 0
    for i in range(1, len(phases)):
        if phases[i] != phases[start]:
            if phases[start] != 0:
                color = 'red' if phases[start] == 1 else 'blue'
                ax.axvspan(dates[start], dates[i - 1], color=color, alpha=alpha)
            start = i
    if phases[start] != 0:
        color = 'red' if phases[start] == 1 else 'blue'
        ax.axvspan(dates[start], dates[-1], color=color, alpha=alpha)

# ── 5-panel figure: N34, N3, N4, N_CT, N_WP ──────────────────────────────────
panel_cfg = [
    ('n34', 'labels_n34', 'Niño3.4',    'steelblue'),
    ('n3',  'labels_n3',  'Niño3',      'darkgreen'),
    ('n4',  'labels_n4',  'Niño4',      'darkorange'),
    ('n_ct','labels_n_ct','N_CT (Cold-Tongue)',  'firebrick'),
    ('n_wp','labels_n_wp','N_WP (Warm-Pool)',    'mediumpurple'),
]

fig, axes = plt.subplots(5, 1, figsize=(15, 14), sharex=True)

for ax, (ik, lk, title, color) in zip(axes, panel_cfg):
    shade_phases(ax, dates, labels_flat[lk])
    ax.plot(dates, indices_flat[ik], color=color, linewidth=0.8)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--', alpha=0.4)
    ax.set_ylabel('Index', fontsize=10)
    ax.set_title(f'{title}  (member {ens_member})', fontsize=11)
    ax.tick_params(labelsize=9)

axes[-1].set_xlabel('Year', fontsize=11)
fig.suptitle(f'CESM2-LE ENSO CP/TP Indices — Member {ens_member}  ({years[0]}–{years[-1]})',
             fontsize=13, y=1.005)
fig.tight_layout()
plt.show()

## 4. Time Series — Selected Month

CP/TP indices for a single calendar month.  
Bars coloured by phase: red = positive, blue = negative, grey = neutral.

In [ ]:
m_idx    = plot_month - 1
mon_name = MONTH_NAMES[m_idx]

def phase_colors(phases):
    return ['red' if p == 1 else 'blue' if p == -1 else 'lightgray' for p in phases]

# Focus on the key CP/TP indices: N_CT and N_WP
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for ax, ik, lk, title, color in [
    (axes[0], 'n_ct', 'labels_n_ct', f'N_CT (Cold-Tongue) — {mon_name}', 'firebrick'),
    (axes[1], 'n_wp', 'labels_n_wp', f'N_WP (Warm-Pool)   — {mon_name}', 'mediumpurple'),
]:
    cols = phase_colors(labels[lk][:, m_idx])
    ax.bar(years, indices[ik][:, m_idx], color=cols, alpha=0.85, width=0.85, edgecolor='none')
    ax.axhline(0, color='black', linewidth=0.7)
    ax.set_ylabel('Index (normalised)', fontsize=11)
    ax.set_title(f'{title}  (member {ens_member})', fontsize=12)
    ax.tick_params(labelsize=10)

axes[-1].set_xlabel('Year', fontsize=11)
fig.suptitle(f'CESM2-LE ENSO CP/TP — {mon_name} — Member {ens_member}',
             fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

### 4b. All five indices — selected month

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)

for ax, (ik, lk, title, color) in zip(axes, panel_cfg):
    cols = phase_colors(labels[lk][:, m_idx])
    ax.bar(years, indices[ik][:, m_idx], color=cols, alpha=0.8, width=0.85, edgecolor='none')
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_ylabel('Index', fontsize=10)
    ax.set_title(f'{title} — {mon_name}  (member {ens_member})', fontsize=11)
    ax.tick_params(labelsize=9)

axes[-1].set_xlabel('Year', fontsize=11)
fig.suptitle(f'CESM2-LE ENSO CP/TP — {mon_name} — Member {ens_member}',
             fontsize=13, y=1.005)
fig.tight_layout()
plt.show()

from src.data.cesm2le.climate_indices import load_grid_latlon

# ── Load grid lat/lon ────────────────────────────────────────────────────────
print('Loading grid lat/lon ...')
lat_grid, lon_grid = load_grid_latlon(GRID_FILE)
print(f'  Grid shape: {lat_grid.shape}   lon range: [{lon_grid.min():.1f}, {lon_grid.max():.1f}]')


def load_member_sst(member_idx, sst_dir):
    """
    Load SST for one ensemble member across all 12 calendar months.
    Returns sst (ntime, nlat, nlon).
    Members 0–49 → first50members files; 50–99 → last50members files.
    """
    if member_idx < 50:
        group, grp_idx = 'first50', member_idx
    else:
        group, grp_idx = 'last50', member_idx - 50

    monthly = []
    for m_label in MONTH_LABELS:
        fpath = sst_dir / f'sst_cesmle_{group}members_mon_{m_label}_199001-210012.nc'
        ds    = nc.Dataset(str(fpath), 'r')
        ds.set_auto_mask(False)
        sst_m = ds.variables['sst_mon'][grp_idx].astype(np.float32)  # (nyear, nlat, nlon)
        ds.close()
        monthly.append(sst_m)
        print(f'  Loaded {m_label}', end='\r')

    # (nyear, 12, nlat, nlon) → (ntime, nlat, nlon) chronologically
    sst_stack = np.stack(monthly, axis=1)
    ny, _, nlat, nlon = sst_stack.shape
    sst = sst_stack.reshape(ny * 12, nlat, nlon)
    print(f'  SST loaded: shape {sst.shape}')
    return sst


print(f'Loading SST for member {ens_member}...')
sst = load_member_sst(ens_member, SST_DIR)

In [ ]:
proj = ccrs.PlateCarree(central_longitude=180)

fig, axes = plt.subplots(2, 2, figsize=(20, 10), subplot_kw={'projection': proj})

panels = [
    (axes[0, 0], composites['n_ct']['pos'], f'N_CT+  (n = {composites["n_ct"]["n_pos"]})'),
    (axes[0, 1], composites['n_ct']['neg'], f'N_CT−  (n = {composites["n_ct"]["n_neg"]})'),
    (axes[1, 0], composites['n_wp']['pos'], f'N_WP+  (n = {composites["n_wp"]["n_pos"]})'),
    (axes[1, 1], composites['n_wp']['neg'], f'N_WP−  (n = {composites["n_wp"]["n_neg"]})'),
]

for ax, data, title in panels:
    ax.set_global()
    ax.coastlines(linewidth=0.5, color='k')
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=2)

    im = ax.pcolormesh(lon_grid, lat_grid, data,
                       cmap=CMAP_SST,
                       vmin=clim_vmin, vmax=clim_vmax,
                       shading='auto',
                       transform=ccrs.PlateCarree())

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(title, fontsize=12, pad=6)

# Row labels
for row_ax, label in [(axes[0, 0], 'Cold-Tongue (N_CT)'), (axes[1, 0], 'Warm-Pool (N_WP)')]:
    row_ax.text(-0.08, 0.5, label, transform=row_ax.transAxes,
                fontsize=12, rotation=90, va='center', ha='center')

fig.subplots_adjust(bottom=0.1, hspace=0.3)
cbar_ax = fig.add_axes([0.25, 0.04, 0.5, 0.025])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
cbar.set_label('SST Anomaly (K)', fontsize=11)

fig.suptitle(
    f'SST composites — ENSO CP/TP phases\n'
    f'CESM2-LE member {ens_member},  {years[0]}–{years[-1]}',
    fontsize=13
)
plt.show()

In [ ]:
proj = ccrs.PlateCarree(central_longitude=180)

fig, axes = plt.subplots(2, 2, figsize=(20, 10), subplot_kw={'projection': proj})

panels = [
    (axes[0, 0], composites['n_ct']['pos'], f'N_CT+  (n = {composites["n_ct"]["n_pos"]})'),
    (axes[0, 1], composites['n_ct']['neg'], f'N_CT−  (n = {composites["n_ct"]["n_neg"]})'),
    (axes[1, 0], composites['n_wp']['pos'], f'N_WP+  (n = {composites["n_wp"]["n_pos"]})'),
    (axes[1, 1], composites['n_wp']['neg'], f'N_WP−  (n = {composites["n_wp"]["n_neg"]})'),
]

for ax, data, title in panels:
    ax.set_global()
    ax.coastlines(linewidth=0.5, color='k')
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=2)

    im = ax.pcolormesh(lon, lat, data,
                       cmap=CMAP_SST,
                       vmin=clim_vmin, vmax=clim_vmax,
                       shading='auto',
                       transform=ccrs.PlateCarree())

    gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5)
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(title, fontsize=12, pad=6)

# Row labels
for row_ax, label in [(axes[0, 0], 'Cold-Tongue (N_CT)'), (axes[1, 0], 'Warm-Pool (N_WP)')]:
    row_ax.text(-0.08, 0.5, label, transform=row_ax.transAxes,
                fontsize=12, rotation=90, va='center', ha='center')

fig.subplots_adjust(bottom=0.1, hspace=0.3)
cbar_ax = fig.add_axes([0.25, 0.04, 0.5, 0.025])
cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
cbar.set_label('SST Anomaly (K)', fontsize=11)

fig.suptitle(
    f'SST composites — ENSO CP/TP phases\n'
    f'CESM2-LE member {ens_member},  {years[0]}–{years[-1]}',
    fontsize=13
)
plt.show()